# miniLLM 49M — bruit entre graines et learning rate plus bas (recette C)

**Deux questions, un seul tableau :**
1. **Quel est le bruit ?** C relancé avec 2 autres graines (`Cs1`, `Cs2`). L'écart entre `C`, `Cs1` et `Cs2` dit à partir de quelle différence on peut croire une comparaison.
2. **Un learning rate plus bas fait-il moins oublier ?** C avec `lr` = 5e-5 (`Clr5`) et 3e-5 (`Clr3`). Tout le reste est identique (mêmes données `sft_C`, 2 époques, même planning).

Nouveau : un **test de connaissances de 204 questions** tenues hors des données SFT (avec intervalle de confiance), à la place des 36 questions de `basics`.

Durée : environ 45 min sur T4 (4 entraînements de ~7 min + 6 évaluations). **Exécution → GPU (T4) → Tout exécuter.** Chaque étape reprend où elle s'est arrêtée si la session coupe.

In [ ]:
# ── 1) Drive, code, vérifications ──
import os, sys, glob, shutil, json
from google.colab import drive
drive.mount("/content/drive")

cands = (["/content/drive/MyDrive/MiniLLM_v2"] + glob.glob("/content/drive/MyDrive/*/MiniLLM_v2")
         + glob.glob("/content/drive/Shareddrives/*/MiniLLM_v2"))
BASE = next((c for c in cands if os.path.isdir(f"{c}/data")), None)
if BASE is None:
    raise SystemExit("Dossier MiniLLM_v2 introuvable (Drive > Partagés > appui long > Ajouter un raccourci > Mon Drive).")
DATA, PT_OUT, CKPT = f"{BASE}/data", f"{BASE}/checkpoints/pretrain", f"{BASE}/checkpoints"
C_DATA, CODE_DIR, LOCAL = f"{DATA}/sft_C", "/content/minillm_v2", "/content/minillm_local"

!pip -q install tokenizers datasets huggingface_hub
if os.path.exists(f"{CODE_DIR}/.git"):
    !git -C {CODE_DIR} fetch -q origin v2.1.1 && git -C {CODE_DIR} checkout -q v2.1.1 && git -C {CODE_DIR} pull -q --ff-only
else:
    !git clone -q -b v2.1.1 https://github.com/bono-p/minillm_v2.git {CODE_DIR}
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)

for p in (f"{PT_OUT}/best.pt", f"{CKPT}/sft_C/final.pt", f"{C_DATA}/meta.json", f"{DATA}/tokenizer.json"):
    print("✓" if os.path.exists(p) else "✗ MANQUANT", p)

import knowledge_eval as ke
print(f"\nTest de connaissances : {len(ke.KNOWLEDGE)} questions | présentes dans les générateurs du SFT : {len(ke.leakage())} (doit être 0)")


In [ ]:
# ── 2) 4 entraînements : 2 graines + 2 learning rates plus bas (~7 min chacun) ──
os.makedirs(LOCAL, exist_ok=True)
shutil.copy(f"{DATA}/tokenizer.json", f"{LOCAL}/tokenizer.json")
local = f"{LOCAL}/sft_C"
if os.path.exists(local):
    shutil.rmtree(local)
shutil.copytree(C_DATA, local)

BASE_ARGS = "--epochs 2 --weight_decay 0.1 --dropout 0.1 --warmup_iters 100 --patience 0 --eval_every 250 --eval_every_dense 100 --eval_dense_until 1500 --save_every 500"
RUNS = {                                   # nom : (lr, min_lr, graine) ; C d'origine = lr 1e-4, graine par défaut (1337)
    "Cs1":  ("1e-4", "1e-5", 1),
    "Cs2":  ("1e-4", "1e-5", 2),
    "Clr5": ("5e-5", "5e-6", 1337),
    "Clr3": ("3e-5", "3e-6", 1337),
}
for name, (lr, min_lr, seed) in RUNS.items():
    print("\n" + "=" * 70 + f"\nEntraînement {name} (lr {lr}, graine {seed})\n" + "=" * 70)
    !python train.py --mode sft --data_dir {local} --out_dir {CKPT}/sft_{name} --init_from {PT_OUT}/best.pt \
        --lr {lr} --min_lr {min_lr} --seed {seed} {BASE_ARGS}


In [ ]:
# ── 3) Tableau comparatif : modèle publié (PUB) + C + 4 variantes (même validation = celle de C, final.pt) ──
from huggingface_hub import hf_hub_download
PUB = "/content/pub"
os.makedirs(PUB, exist_ok=True)
for f in ("best.pt", "tokenizer.json"):
    hf_hub_download("bonopassale/miniLLM_v2.1-49M-42500it_2.79Btoks_0.9ep_20261001", f, local_dir=PUB)
specs = ["--ckpt " + f"PUB={PUB}/best.pt", "--ckpt " + f"C={CKPT}/sft_C/final.pt"] + ["--ckpt " + f"{n}={CKPT}/sft_{n}/final.pt" for n in RUNS]
SPECS = " ".join(specs)
os.makedirs(f"{BASE}/evals_lr_seeds", exist_ok=True)
!python compare_sft.py {SPECS} --val_dir {C_DATA} --out_dir {BASE}/evals_lr_seeds --qa_n 300 --tokenizer {DATA}/tokenizer.json --noise_group C,Cs1,Cs2


**Comment lire le résultat**
- La section **« Bruit entre graines »** donne l'étendue de chaque mesure sur `C`, `Cs1`, `Cs2`. Le learning rate plus bas ne compte que si `Clr5` / `Clr3` sortent **nettement** de cette étendue (et de l'intervalle `know±`).
- `know±` : taux de réussite sur 204 questions et demi-largeur de l'intervalle de confiance à 95 %.
- `PUB` = le modèle publié sur Hugging Face (v2.1) : sur `qa`, il a pu voir des exemples de cette validation, donc la comparaison avec lui est à lire avec prudence ; `know±` et `persona` ne souffrent pas de ce biais.
- Si rien ne sort du bruit : la recette C est stable, et ni un learning rate plus bas ni une LoRA ne changeront la qualité.

**À m'envoyer :** le tableau et la section « Bruit entre graines ».